# broadcast-source-fanout — ex2: nn.Embedding lookup with verified sparse gradient on absent classes

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `broadcast-source-fanout`. Running the final beacon cell reports progress against the `Generative: Broadcast source fan-out` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: Broadcast source fan-out` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`broadcast-source-fanout`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "broadcast-source-fanout"
DD_SUBTOPIC = "Generative: Broadcast source fan-out"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `nn.Embedding` and sparse gradients — quick refresher

```python
embed = nn.Embedding(num_classes, D)
out = embed(labels)        # labels: (B,) -> out: (B, D)
```

`nn.Embedding` is a thin wrapper: `weight` is a `(num_classes, D)` `Parameter`, forward is `F.embedding(labels, self.weight)` (the same integer-indexing as ex1 — `weight[labels]`). The Module form gets you `.weight` as a registered parameter, so the optimizer sees it automatically.

**Sparse gradient property.** When you backprop through `embed(labels)`, the gradient is non-zero ONLY for the rows of `weight` that were actually indexed. Classes absent from the batch get exact zero — not a tiny floating-point value, but a structural zero from the autograd graph itself.

**Why this matters.** With imbalanced label distributions, rare classes accumulate updates SLOWLY because they're indexed rarely. `optim.SparseAdam` is built for exactly this case (only updates the rows that received non-zero grad), saving compute when num_classes is huge.

### Exercise 2 — nn.Embedding lookup with verified sparse gradient on absent classes

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Analyze
> LO: Analyze the embedding-lookup gradient structure by building an `nn.Embedding`, fanning out via labels, summing the result, calling backward, and verifying that ONLY rows for classes present in the batch receive non-zero gradient.
> Keywords: nn.Embedding, sparse-grad, absent-classes, F.embedding
> ```

**KCs targeted:** `nn-embedding-equiv-to-index`, `sparse-grad-on-unindexed-rows`

Implement `ex2_embed_with_sparse_grad(num_classes, D, labels)`. Build the Module form, do the fan-out, force a backward, then return the embedding's `weight.grad` so the caller can inspect the sparsity:

1. Construct `embed = nn.Embedding(num_classes, D)`. Initialize `embed.weight.data` to a known matrix: `t.arange(num_classes * D, dtype=t.float32).reshape(num_classes, D)`. This gives every row a unique fingerprint.
2. Fan out: `per_sample = embed(labels)` (shape `(B, D)`).
3. Compute a scalar loss: `loss = per_sample.sum()`. (Sum of all elements — gives gradient 1 per element of `per_sample`.)
4. Call `loss.backward()`.
5. Return a tuple `(per_sample.detach(), embed.weight.grad.clone())`. Detach the forward so the caller can read it without complicating its own autograd graph; clone the grad so future backward calls don't mutate the returned tensor.

What the gradient looks like. Each row of `weight.grad` equals the number of times that class appeared in `labels` (because `loss = sum` differentiates element-wise to 1, so each fan-out contributes 1 to each component of the row it indexed). A class absent from labels has `weight.grad[c]` exactly zero (structural zero from autograd, not just numerically zero).

In [ ]:
def ex2_embed_with_sparse_grad(num_classes: int, D: int, labels: Tensor):
    import torch.nn as nn
    embed = nn.Embedding(num_classes, D)
    with t.no_grad():
        embed.weight.copy_(t.arange(num_classes * D, dtype=t.float32).reshape(num_classes, D))
    per_sample = embed(labels)
    loss = per_sample.sum()
    loss.backward()
    return per_sample.detach(), embed.weight.grad.clone()


<details><summary>Solution</summary>

```python
def ex2_embed_with_sparse_grad(num_classes: int, D: int, labels: Tensor):
    import torch.nn as nn
    embed = nn.Embedding(num_classes, D)
    with t.no_grad():
        embed.weight.copy_(t.arange(num_classes * D, dtype=t.float32).reshape(num_classes, D))
    per_sample = embed(labels)
    loss = per_sample.sum()
    loss.backward()
    return per_sample.detach(), embed.weight.grad.clone()
```

**`nn.Embedding(labels) == weight[labels]`.** They produce bit-identical tensors and the same autograd graph. The Module form's only difference: `weight` is auto-registered as a `nn.Parameter`, so an optimizer over `embed.parameters()` picks it up.

**Why the absent rows are structural zero.** Autograd builds the gradient as `sum over batch of (one-hot[labels[i]] · d(loss)/d(per_sample[i]))`. The one-hot is zero for every class that didn't appear, so no contribution accumulates there. This is genuinely zero — not 1e-12 epsilon noise.

**SparseAdam exploits this.** For embedding tables with millions of classes (token embeddings in a large vocab), most rows have zero grad most steps. `optim.SparseAdam` reads `weight.grad` as a sparse tensor and skips the zero rows — linear-time-per-step independent of vocab size. The drill doesn't use SparseAdam, but the sparsity property is the same.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()